# Chest CT Nodules

This notebook is the lightweight EDA/evaluation companion for `src/ctscan`.

Use it to inspect the real LIDC subset patch dataset, retrain the shipped PyTorch checkpoint, and review simple evaluation metrics before launching the app.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from src.ctscan.models.nodules import load_model_bundle
from src.ctscan.scripts.nodules.train_model import load_training_dataset

## Prepare data

Run this once from `src/ctscan` if the real LIDC subset and TCIA sample studies are not present yet:

```bash
python scripts/nodules/download_data.py --lidc-study-limit 7
```

In [ ]:
root = Path("src/ctscan")
dataset_path = root / "data" / "ctscan" / "processed" / "nodules_training.npz"
patches, nodule_target, malignancy_target = load_training_dataset(dataset_path)
patches.shape, nodule_target.shape, malignancy_target.shape

In [ ]:
summary = pd.DataFrame(
    {
        "rows": [int(patches.shape[0])],
        "positive_nodule_rate": [float(nodule_target.mean())],
        "positive_malignancy_rate": [float(malignancy_target.mean())],
        "patch_mean_hu": [float(patches.mean())],
        "patch_std_hu": [float(patches.std())],
    }
)
summary

## Train / refresh the checkpoint

```bash
python scripts/nodules/train_model.py --model-version 0.3.0
```

In [ ]:
model_path = root / "models" / "nodules.pt"
model, patch_mean, patch_std, version, metrics = load_model_bundle(model_path)
version, metrics

## Notes

- The committed checkpoint is trained on a real `LIDC-IDRI` subset produced by `download_data.py` with `7` annotated studies.
- The deterministic smoke dataset is still generated as a fallback for tests and for rebuilding the repo without hitting the public APIs.
- Scale `--lidc-study-limit` upward before treating the reported metrics as representative.
- Local app launch stays simple:

```bash
cd src/ctscan
python main.py
```